## **Problem Statement**

### **Business Context**

Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets, which protect against head injuries from falling objects and machinery. Non-compliance with helmet regulations increases the risk of serious injuries or fatalities, making effective monitoring essential, especially in large-scale operations where manual oversight is prone to errors and inefficiency.

To overcome these challenges, SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets. This system will improve safety enforcement, ensuring compliance and reducing the risk of head injuries. By automating helmet monitoring, SafeGuard aims to enhance efficiency, scalability, and accuracy, ultimately fostering a safer work environment while minimizing human error in safety oversight.

### **Objective**

As a data scientist at SafeGuard Corp, you are tasked with developing an image classification model that classifies images into one of two categories:
- **With Helmet:** Workers wearing safety helmets.
- **Without Helmet:** Workers not wearing safety helmets.

### **Data Description**

The dataset consists of **4125 images**, divided into two categories:

- **With Helmet:** 3161 images showing workers wearing helmets.
- **Without Helmet:** 964 images showing workers not wearing helmets.

**Dataset Characteristics:**
- **Variations in Conditions:** Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
- **Worker Activities:** Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

## **Installing and Importing the Necessary Libraries**

In [ ]:
# !pip install tensorflow[and-cuda] scikit-learn==1.6.1 opencv-python==4.12.0.88 seaborn==0.13.2 matplotlib==3.10.0 numpy==2.0.2 pandas==2.2.2 -q

# pip install tensorflow-macos tensorflow-metal
# pip install "numpy<2.0.0,>=1.26.0"
# pip install \
#     scikit-learn==1.6.1 \
#     opencv-python-headless==4.12.0.88 \
#     seaborn==0.13.2 \
#     matplotlib==3.10.0 \
#     pandas==2.2.2 \
#     scipy==1.17.1 \
#     --no-deps



In [1]:
# Check TensorFlow version and GPU availability
import tensorflow as tf

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# Check if TensorFlow is using the GPU
if tf.config.list_physical_devices("GPU"):
    print("TensorFlow is using the GPU.")
else:
    print("TensorFlow is not using the GPU.")
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

TF: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow is using the GPU.
Num GPUs Available: 1


In [2]:
import os
import random
import numpy as np                                                                               # Importing numpy for Matrix Operations
import pandas as pd
import seaborn as sns
import matplotlib.image as mpimg                                                                              # Importing pandas to read CSV files
import matplotlib.pyplot as plt                                                                  # Importting matplotlib for Plotting and visualizing images
import math                                                                                      # Importing math module to perform mathematical operations
import cv2


# Tensorflow modules
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator                            # Importing the ImageDataGenerator for data augmentation
from tensorflow.keras.models import Sequential                                                   # Importing the sequential module to define a sequential model
from tensorflow.keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D,BatchNormalization # Defining all the layers to build our CNN Model
from tensorflow.keras.optimizers import Adam,SGD                                                 # Importing the optimizers which can be used in our model
from sklearn import preprocessing                                                                # Importing the preprocessing module to preprocess the data
from sklearn.model_selection import train_test_split                                             # Importing train_test_split function to split the data into train and test
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight                             # Importing compute_class_weight to compute class weights for imbalanced datasets
from keras.applications.vgg16 import VGG16                                               # Importing confusion_matrix to plot the confusion matrix

#Imports functions for evaluating the performance of machine learning models
from sklearn.metrics import confusion_matrix, f1_score,accuracy_score, recall_score, precision_score, classification_report

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress TF logs

In [ ]:
# Set random seeds for reproducibility
SEED = 812

# 1. Set Python random seed for dataset shuffling, sampling
random.seed(SEED)

# 2. Set NumPy random seed for preprocessing, augmentations, and any NumPy-based operations
np.random.seed(SEED)

# 3. Set TensorFlow seed (covers Keras + backend) for weight init, dropout, training, and any TF-based operations
tf.keras.utils.set_random_seed(SEED)

# 4. Enable deterministic TensorFlow operations
tf.config.experimental.enable_op_determinism()

## **Data Overview**


### Loading the data

In [ ]:
# Load the dataset
images = np.load("images.npy")
labels_df = pd.read_csv("labels.csv")

labels = labels_df["label"].values

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print(labels_df.head())
print(labels_df["label"].value_counts())

**Observation**

The dataset consists of 4,125 RGB images (200×200 pixels) with corresponding labels, showing a class imbalance with 3,161 “With Helmet” and 964 “Without Helmet” samples.

In [ ]:
# Define class names for better readability in confusion matrix and classification report

class_names = {
    0: "Without Helmet",
    1: "With Helmet"
}

# **Exploratory Data Analysis**

### Plot random images from each of the classes and print their corresponding labels.

In [ ]:
# Visualize some random images from the dataset
plt.figure(figsize=(12, 6))

for i in range(10):
    idx = random.randint(0, len(images) - 1)
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[idx])
    plt.title(f"{class_names[labels[idx]]}\nIdx: {idx}")
    plt.axis("off")

plt.tight_layout()
plt.show()

**Observations**

* The dataset shows **real-world variability (lighting, angles, environments)**, which is beneficial for building a model suitable for **robust, real-time deployment**
* Presence of **blurred and partially visible workers/helmets** highlights the need for a model that can handle **imperfect visual conditions in safety monitoring**
* Some **ambiguous or noisy samples** may impact classification accuracy, reinforcing the importance of **using advanced models (transfer learning)**


## Plotting class imbalance


In [ ]:
# Check class distribution

plt.figure(figsize=(6, 4))

ax = sns.countplot(x=labels, palette=['red', 'green'])

# Apply class names
ax.set_xticklabels([class_names[i] for i in sorted(class_names)])

# Add count + %
total = len(labels)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}\n({p.get_height()/total:.1%})',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

**Observation**

* The dataset is imbalanced. The With Helmet class has 3161(76.6%) images, while Without Helmet has 964(23.4%) images. This means accuracy alone may be misleading, so precision, recall, F1-score, and confusion matrix should be used when comparing models.
* The noticeable **class imbalance** suggests the model must be carefully evaluated to ensure it reliably detects **“Without Helmet” cases**, which are critical for safety enforcement

## **Data Preprocessing**

Based on the EDA, several key characteristics of the dataset influence the preprocessing approach:

| Step                        | What to Do                                                                     | Why (Based on EDA)                                                                                                     |
| --------------------------- | ------------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------------------------------- |
| Train/Validation/Test Split | Split data into train, validation, and test sets using **stratified sampling** | Dataset is **imbalanced**, so stratification ensures each split maintains the same class distribution                  |
| Normalization               | Scale pixel values from **0–255 to 0–1**                                       | Images have **varying lighting and quality**, normalization helps the model learn more effectively and converge faster |
| Validation Set Creation     | Keep a separate validation set during training                                 | Presence of **noisy and ambiguous samples** increases risk of overfitting; validation helps monitor generalization     |
| Test Set Reservation        | Hold out a final test set for evaluation                                       | Ensures **unbiased performance evaluation** on unseen data, simulating real-world deployment conditions                |


### Splitting the dataset



In [ ]:
# Split the dataset into training, validation, and test sets

# Ensure correct types 
X_raw = np.asarray(images, dtype="float32")
y = np.asarray(labels, dtype="int32")

# 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Shapes
for name, arr in {
    "X_train": X_train, "X_val": X_val, "X_test": X_test,
    "y_train": y_train, "y_val": y_val, "y_test": y_test
}.items():
    print(f"{name}: {arr.shape}")

**Observation**

* The dataset has been successfully split into training, validation, and test sets using stratification. 
* The shapes confirm that the majority of data is allocated to training, with equal proportions for validation and test sets. 
* Class distribution is preserved across all splits, ensuring unbiased model training and evaluation.

### Data Normalization

In [ ]:
# Normalise (0–255 → 0–1) split datasets by scaling pixel values to the range [0, 1] 
X_train_norm = X_train.astype("float32") / 255.0
X_val_norm   = X_val.astype("float32") / 255.0
X_test_norm  = X_test.astype("float32") / 255.0

# Ensure labels dtype
y_train = y_train.astype("int32")
y_val   = y_val.astype("int32")
y_test  = y_test.astype("int32")

# Check normalization
print("Train min/max:", X_train_norm.min(), X_train_norm.max())
print("Validation min/max:", X_val_norm.min(), X_val_norm.max())
print("Test min/max:", X_test_norm.min(), X_test_norm.max())    

## **Model Building**

After splitting the dataset and normalizing the image pixel values, the next step is to build and evaluate different image classification models.

The objective is to classify each worker image as either **With Helmet** or **Without Helmet**. Since the EDA showed class imbalance and image variability, multiple models will be trained and compared to identify the most reliable model for workplace safety monitoring.

The models will be evaluated using validation performance, confusion matrix, precision, recall, F1-score, and false positive rate. This is important because misclassifying a worker without a helmet as wearing one could create serious safety risks.

### Models Overview

Four models are developed in a progressive manner, starting with a simple CNN baseline and advancing to more sophisticated transfer learning approaches using VGG16. This structured approach enables a clear comparison of how feature extraction, model complexity, and data augmentation contribute to improving classification performance and robustness in detecting helmet compliance.

| Model                            | Description                                               | Purpose                                     | Expected Outcome                             |
| -------------------------------- | --------------------------------------------------------- | ------------------------------------------- | -------------------------------------------- |
| CNN from Scratch                 | Custom-built convolutional neural network                 | Establish a baseline for comparison         | Moderate performance, prone to overfitting   |
| VGG16 (Base)                     | Pretrained VGG16 with frozen layers + simple output layer | Leverage pretrained feature extraction      | Improved accuracy over baseline              |
| VGG16 + FFNN                     | VGG16 base with additional dense and dropout layers       | Enhance task-specific learning              | Better precision, recall, and generalization |
| VGG16 + FFNN + Data Augmentation | Previous model with image augmentation techniques         | Improve robustness to real-world variations | Best overall performance and generalization  |

This progression ensures that model improvements are not arbitrary but driven by specific enhancements addressing dataset limitations such as variability, noise, and class imbalance.

In [ ]:
# initialize dictionaries to store model results and trained models for later comparison and analysis

models_dict = {}
model_results = []

In [ ]:
# Compute class weights to address class imbalance in the training data, ensuring that the model pays more attention to the minority class 
# during training and improves performance on underrepresented classes.

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights_array))
print(class_weights)

In [ ]:
# Define common configuration for model compilation and training to ensure consistency across different models and simplify code maintenance

common_config = {
    "loss": "binary_crossentropy",
    "optimizer": "adam",
    "batch_size": 32,
    "epochs": 50,
        "class_weights": class_weights
}

# The same loss function and optimizer are used across all models for consistency.
# All models were trained with the same maximum number of epochs to ensure fair comparison. 
# Early stopping was used to prevent overfitting and allow each model to converge at its optimal point.

In [ ]:
# Define early stopping callback to prevent overfitting and save best model weights

# Early stopping monitors validation loss as it provides a more sensitive measure of model generalisation compared to accuracy, helping to 
# prevent overfitting and ensure optimal performance on unseen data.

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5, # Adjusted patience to 5 to allow for quicker convergence while still preventing overfitting
    restore_best_weights=True
)

### Model Evaluation Criterion

* Each model will be evaluated using the same criteria to ensure a fair comparison. Since the objective is to detect whether workers are wearing helmets, the model must perform well overall while also correctly identifying workers without helmets, as this is the highest-risk safety case.

* Class weights were applied during training to address class imbalance and give higher importance to the minority “Without Helmet” class. **This is important because correctly identifying workers without helmets is the main safety-critical objective.**

| Metric | What It Measures | Why It Matters | Priority |
|---|---|---|---|
| Recall (Without Helmet) | Proportion of actual “Without Helmet” cases correctly identified | Ensures unsafe workers are detected and not missed | Highest |
| False Positive Rate (FPR) | Proportion of “Without Helmet” cases incorrectly predicted as “With Helmet” | Measures how often unsafe workers are wrongly classified as safe | Critical |
| F1-score | Balance between precision and recall | Useful for handling class imbalance and overall performance balance | High |
| Precision | Proportion of correct positive predictions | Helps assess reliability of helmet/non-helmet predictions | Medium |
| Accuracy | Overall proportion of correct predictions | Provides general performance but can be misleading due to imbalance | Low |
| Confusion Matrix | Breakdown of predictions by class | Helps identify types of errors (FP, FN) clearly | Supporting |
| Validation Loss/Accuracy Curves | Model performance over training epochs | Helps detect overfitting or underfitting | Supporting |

### Utility Functions

In [ ]:
# function to visualize predictions of the model 

def visualize_predictions(model, X_data, y_data, class_names, n=10):
    """
    Display sample images with actual and predicted labels.
    """
    y_prob = model.predict(X_data)
    y_pred = (y_prob > 0.5).astype("int32").reshape(-1)
    y_true = np.asarray(y_data).reshape(-1)

    indices = np.random.choice(len(X_data), size=n, replace=False)

    plt.figure(figsize=(15, 6))

    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i + 1)
        plt.imshow(X_data[idx])

        actual = class_names[y_true[idx]]
        predicted = class_names[y_pred[idx]]

        plt.title(
            f"Actual: {actual}\nPred: {predicted}",
            fontsize=9
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# function to evaluate model performance

def evaluate_model(model, X_data, y_data, model_name):

    tf.keras.backend.clear_session()  # Reset graph to avoid clutter from previous models

    y_prob = model.predict(X_data)
    y_pred = (y_prob > 0.5).astype("int32").reshape(-1)
    y_true = np.asarray(y_data).reshape(-1)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    recall_without = recall_score(y_true, y_pred, pos_label=0)
    precision_without = precision_score(y_true, y_pred, pos_label=0)
    f1_without = f1_score(y_true, y_pred, pos_label=0)

    print(f"\nModel: {model_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall (Without Helmet): {recall_without:.4f}")
    print(f"Precision (Without Helmet): {precision_without:.4f}")
    print(f"F1 (Without Helmet): {f1_without:.4f}")
    print(f"FPR: {fpr:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["Without Helmet", "With Helmet"]
    ))

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Without Helmet", "With Helmet"],
        yticklabels=["Without Helmet", "With Helmet"]
    )
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return accuracy, fpr, recall_without, precision_without, f1_without

In [ ]:
# Function to plot training history (accuracy and loss)

def plot_training_history(history, model_name):
    """
    Plot training and validation accuracy/loss for a trained model.
    """
    plt.figure(figsize=(12, 4))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

### Model 1: Convolutional Neural Network (CNN) from Scratch

This CNN model is built from scratch and used as the baseline model. It has two main parts:

1. **Feature Extraction Layers** – convolutional and pooling layers that learn visual patterns from the images.
2. **Classification Layers** – fully connected layers that use the extracted features to predict either **With Helmet** or **Without Helmet**.

In [ ]:
# Define a simple CNN model from scratch

# 1. Define model
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(200, 200, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

# 2. Compile (no assignment)
cnn_model.compile(
    optimizer=common_config["optimizer"],
    loss=common_config["loss"],
    metrics=["accuracy"]
)

# 3. Train
history_cnn = cnn_model.fit(
    X_train_norm,
    y_train,
    validation_data=(X_val_norm, y_val),
    epochs=common_config["epochs"],
    batch_size=common_config["batch_size"],
    callbacks=[early_stop],
    class_weight=common_config["class_weights"]
)

In [ ]:
#  4. Evaluate the CNN model on the validation set and store results

plot_training_history(history_cnn, "CNN from Scratch")

cnn_accuracy, cnn_fpr, cnn_recall, cnn_precision, cnn_f1 = evaluate_model(
    cnn_model,
    X_val_norm,
    y_val,
    "CNN from Scratch"
)

**CNN from Scratch – Performance Observation**

- The model achieves a **high overall accuracy (93.21%)**, indicating strong general performance on the validation set.  
- **Recall for “Without Helmet” is 0.94**, meaning the majority of unsafe workers are correctly identified, which is critical for safety monitoring.  
- The **False Positive Rate (5.56%) is low**, showing that only a small proportion of unsafe cases are incorrectly classified as safe.  
- From the confusion matrix, only **8 unsafe cases were misclassified**, demonstrating good performance on the minority class.

**Training Behaviour**

- Training and validation accuracy follow a similar trend, suggesting **good generalisation**.  
- Both training and validation loss decrease and stabilise, indicating **stable learning without significant overfitting**.  
- Performance converges within a few epochs, showing the model is learning efficiently from the data.

**Key Insight**

The CNN from Scratch provides a strong baseline, performing well on both overall accuracy and the priority safety metrics (**recall and FPR for “Without Helmet”**). This establishes a solid reference point for evaluating the impact of transfer learning in subsequent models.

#### Visualizing the predictions

In [ ]:
visualize_predictions(cnn_model, X_val_norm, y_val, class_names)

**Prediction Visualisation Summary**

- The model correctly classifies most images, including several **blurred and low-quality samples**, indicating it has learned general visual patterns effectively.  
- A few misclassifications still occur, particularly in **ambiguous or unclear cases**, but overall the model shows good reliability in distinguishing between helmet and non-helmet images.

In [ ]:
# 5. Store results and model in the dictionary for later comparison and analysis

models_dict["CNN from Scratch"] = {
    "model": cnn_model,
    "history": history_cnn,
    "accuracy": cnn_accuracy,
    "fpr": cnn_fpr,
    **common_config
}

model_results.append({
    "Model": "CNN from Scratch",
    "Accuracy": cnn_accuracy,
    "Recall (No Helmet)": cnn_recall,
    "Precision (No Helmet)": cnn_precision,
    "F1 (No Helmet)": cnn_f1,
    "FPR": cnn_fpr
})

model_results

### Model 2: Transfer Learning with VGG-16 (Base)

* The CNN from Scratch provided a useful baseline, but it still missed some **Without Helmet** cases, which is risky for a safety monitoring system. To improve feature extraction, the next model uses **VGG16**, a pre-built CNN architecture trained on the ImageNet dataset.

* VGG16 is used as a frozen feature extractor. Its convolutional and pooling layers are kept unchanged, meaning their weights are not updated during training. A simple classification head consisting of a Flatten layer and one Dense output layer is added to predict whether an image belongs to **With Helmet** or **Without Helmet**.

* This approach tests whether pretrained visual features can improve performance over the baseline CNN, especially on the priority metrics: **Recall for Without Helmet** and **False Positive Rate**.

In [ ]:
# Define a transfer learning model using VGG16 as the base

# 1. Define model
vgg16_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(200, 200, 3)
)

# Freeze pretrained layers
vgg16_base.trainable = False

vgg16_base_model = Sequential([
    vgg16_base,
    Flatten(),
    Dense(1, activation="sigmoid")
])

vgg16_base_model.summary()

# 2. Compile
vgg16_base_model.compile(
    optimizer=common_config["optimizer"],
    loss=common_config["loss"],
    metrics=["accuracy"]
)

# 3. Train
history_vgg16_base = vgg16_base_model.fit(
    X_train_norm,
    y_train,
    validation_data=(X_val_norm, y_val),
    epochs=common_config["epochs"],
    batch_size=common_config["batch_size"],
    callbacks=[early_stop],
    class_weight=class_weights
)

In [ ]:
# 4. Evaluate the VGG16 base model on the validation set 

plot_training_history(history_vgg16_base, "VGG16 Base")

vgg16_base_accuracy, vgg16_base_fpr, vgg16_base_recall, vgg16_base_precision, vgg16_base_f1 = evaluate_model(
    vgg16_base_model,
    X_val_norm,
    y_val,
    "VGG16 Base"
)


**VGG16 Base – Performance Observation**

- The model achieves an **overall accuracy of 85.30%**, which is lower than the CNN baseline, indicating weaker general performance.  
- **Recall for “Without Helmet” is 0.80**, meaning a noticeable proportion of unsafe workers are still missed.  
- The **False Positive Rate (20.14%) is relatively high**, showing that many unsafe cases are incorrectly classified as safe.  
- From the confusion matrix, **29 unsafe cases were misclassified**, which is higher than the CNN model, indicating poorer performance on the safety-critical class.

**Training Behaviour**

- Training accuracy increases steadily, but validation accuracy improves slowly and plateaus early, indicating **limited generalisation**.  
- Training loss decreases consistently, while validation loss reduces more gradually, suggesting the model is learning but not effectively transferring knowledge to unseen data.  
- The gap between training and validation performance indicates **mild overfitting**.

**Key Insight**

Although VGG16 provides strong pretrained feature extraction, the simple classification head is insufficient for this task. The model performs worse than the CNN baseline on the priority metrics (**recall and FPR for “Without Helmet”**), highlighting the need for a more expressive classification layer in subsequent models.

#### Visualizing the predictions

In [ ]:
visualize_predictions(vgg16_base, X_val_norm, y_val, class_names)

**Prediction Visualisation Summary**

- The model shows **inconsistent predictions**, with several incorrect classifications in both directions, particularly misclassifying clear helmet and non-helmet cases.  
- Errors are more frequent in **blurred, low-quality, or ambiguous images**, indicating the simple classification head struggles to effectively utilise VGG16 features for this task.

In [ ]:
# 6. Store results and model in the dictionary for later comparison and analysis
models_dict["VGG16 Base"] = {
    "model": vgg16_base_model,
    "history": history_vgg16_base,
    "accuracy": vgg16_base_accuracy,
    "fpr": vgg16_base_fpr,
    **common_config
}

model_results.append({
    "Model": "VGG16 Base",
    "Accuracy": vgg16_base_accuracy,
    "Recall (No Helmet)": vgg16_base_recall,
    "Precision (No Helmet)": vgg16_base_precision,
    "F1 (No Helmet)": vgg16_base_f1,
    "FPR": vgg16_base_fpr
})

model_results

### Model 3: Transfer Learning with VGG-16 (Base + FFNN)





The VGG16 Base model showed that while pretrained features are useful, the simple classification head was not sufficient, leading to poor performance on the priority metrics (Recall and FPR for “Without Helmet”).

In this model, the convolutional and pooling layers of VGG16 are used as a frozen feature extractor. For classification, a **Flatten layer and a Feed Forward Neural Network (FFNN)** are added to improve task-specific learning.


In [ ]:
# Define Model 3: VGG16 Base + FFNN

# 1. Define model
vgg16_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(200, 200, 3)
)

# Freeze convolutional layers
vgg16_base.trainable = False

vgg_ffnn_model = Sequential([
    vgg16_base,
    Flatten(),

    Dense(64, activation="relu"),
    Dropout(0.4),           # Added dropout to reduce overfitting

    Dense(32, activation="relu"),

    Dense(1, activation="sigmoid")
])

vgg_ffnn_model.summary()

# 2. Compile Model 3
vgg_ffnn_model.compile(
    optimizer=Adam(learning_rate=0.0001),  # lower LR for stability
    loss=common_config["loss"],
    metrics=["accuracy"]
)

# 3. Train Model 3

history_vgg_ffnn = vgg_ffnn_model.fit(
    X_train_norm,
    y_train,
    validation_data=(X_val_norm, y_val),
    epochs=common_config["epochs"],
    batch_size=common_config["batch_size"],
    callbacks=[early_stop],
    class_weight=class_weights
)

In [ ]:
# 4. Evaluate the VGG16 + FFNN model on the validation set 

plot_training_history(history_vgg_ffnn, "VGG16 + FFNN")

vgg_ffnn_accuracy, vgg_ffnn_fpr, vgg_ffnn_recall, vgg_ffnn_precision, vgg_ffnn_f1 = evaluate_model(
    vgg_ffnn_model,
    X_val_norm,
    y_val,
    "VGG16 + FFNN"
)

**VGG16 + FFNN – Performance Observation**

- The model achieves an **accuracy of 74.64%**, which is lower than both the CNN baseline and VGG16 Base, indicating weaker overall performance.  
- **Recall for “Without Helmet” improves to 0.85**, meaning more unsafe workers are correctly identified compared to the VGG16 Base model.  
- The **False Positive Rate (14.58%) is lower than VGG16 Base but still relatively high**, indicating a notable number of unsafe cases are incorrectly classified as safe.  
- From the confusion matrix, **21 unsafe cases were misclassified**, showing improvement over earlier VGG16 results but still not matching the CNN baseline.

**Training Behaviour**

- Training accuracy increases steadily, while validation accuracy improves gradually, indicating **stable but limited learning capacity**.  
- Both training and validation loss decrease consistently, suggesting **no major overfitting** and improved training stability compared to earlier runs.  
- The gap between training and validation performance is moderate, indicating **reasonable generalisation**.

**Key Insight**

Adding a deeper FFNN improves the model’s ability to detect the minority class (**Without Helmet**) compared to the VGG16 Base model. However, overall performance and False Positive Rate are still not as strong as the CNN baseline, suggesting that further improvements (e.g., data augmentation) are required to enhance robustness and safety-critical detection.

Model 3 (VGG16 + FFNN) was expected to improve over the VGG16 Base model by adding a stronger FFNN classification head. However, its performance did not exceed the CNN baseline on the priority safety metrics. This suggests that increasing classifier complexity alone is insufficient, and further improvement through data augmentation is required in Model 4.

#### Visualizing the predictions

In [ ]:
# Visualise predictions

visualize_predictions(
    vgg_ffnn_model,
    X_val_norm,
    y_val,
    class_names
)

**Prediction Visualisation Summary**

- The model shows **mixed performance**, with both correct predictions and noticeable errors, including misclassifying some “Without Helmet” cases as “With Helmet” and vice versa.  
- Several errors occur in **blurred or low-clarity images**, suggesting the model still struggles with ambiguous visual features despite the improved classification head.

In [ ]:
# Store Model 3 results

models_dict["VGG16 + FFNN"] = {
    "model": vgg_ffnn_model,
    "history": history_vgg_ffnn,
    "accuracy": vgg_ffnn_accuracy,
    "fpr": vgg_ffnn_fpr,
    **common_config
}
model_results.append({
    "Model": "VGG16 + FFNN",
    "Accuracy": vgg_ffnn_accuracy,
    "Recall (No Helmet)": vgg_ffnn_recall,
    "Precision (No Helmet)": vgg_ffnn_precision,
    "F1 (No Helmet)": vgg_ffnn_f1,
    "FPR": vgg_ffnn_fpr
})

model_results

## Model 4: Transfer Learning with VGG-16 (Base + FFNN + Data Augmentation)

- In most of the real-world case studies, it is challenging to acquire a large number of images and then train CNNs.
- To overcome this problem, one approach we might consider is **Data Augmentation**.
- CNNs have the property of **translational invariance**, which means they can recognise an object even if its appearance shifts translationally in some way. - Taking this attribute into account, we can augment the images using the techniques listed below

    -  Horizontal Flip (should be set to True/False)
    -  Vertical Flip (should be set to True/False)
    -  Height Shift (should be between 0 and 1)
    -  Width Shift (should be between 0 and 1)
    -  Rotation (should be between 0 and 180)
    -  Shear (should be between 0 and 1)
    -  Zoom (should be between 0 and 1) etc.

Remember, **data augmentation should not be used in the validation/test data set**.

#### Visualizing the predictions

# **Model Performance Comparison and Final Model Selection**

In [ ]:
# 5. Create comparison table

model_results_df = pd.DataFrame(model_results)
model_results_df.sort_values(by="Recall (No Helmet)", ascending=False)
model_results_df.round(3)

model_results_df.sort_values(
    by=["Recall (No Helmet)", "FPR"],
    ascending=[False, True]
)

## Test Performance

# **Actionable Insights & Recommendations**

-
-

<font size=5 color='blue'>Power Ahead!</font>
___